# PSA Machine Translation — Week 3: Modeling with Transfer Learning

**Course:** DSA 4020A NLP · Semester Project · **Focus:** Sub-objective 2 (few-shot cross-lingual transfer)

This notebook consumes the **Week 2 outputs** (`data_processed/`) and delivers the Week 3 checklist:

- Experiment tracking (**MLflow**, offline-friendly; optional Weights & Biases)
- **Two** pretrained models with few-shot fine-tuning: **NLLB-200-distilled-600M** and **mT5-small**
- Low-resource handling: **encoder freezing** + few-shot subsampling (+ back-translation hook)
- **Ablation studies**: zero-shot vs few-shot, freeze vs full fine-tune, per-domain
- Saved **checkpoints & logs**, documented **hyperparameters**, and an **initial performance summary**
- A working **inference demo** (notebook function + a `translate_psa.py` CLI)

**Design rationale (important):** NLLB-200 *does not include Ekegusii*, so we use it as a strong
baseline for the supported **English↔Kiswahili** directions. **mT5-small is text-to-text and
language-agnostic**, so it is our workhorse for the low-resource **Ekegusii** directions (the
project's headline goal). Using both satisfies "≥2 models" and directly demonstrates the
low-resource transfer story.

> **Runtime:** built for **Google Colab (GPU)**. Training cells download model weights from
> Hugging Face and need a GPU; the data/metric/inference plumbing runs on CPU. Set `DRY_RUN = True`
> first to smoke-test the whole pipeline in ~1 minute before committing to a full run.

## 0 · Environment, dependencies & hardware

The first cell **auto-installs** anything missing (Colab or local), importing only what isn't
already present — so you never hit a `ModuleNotFoundError`. If Colab says "Restart session" after
installing, do it, then run this cell again (it will skip the install and continue).

In [1]:
# --- auto-install missing dependencies into THIS kernel -----------------------
import importlib.util, subprocess, sys

REQUIREMENTS = {
    "transformers": "transformers>=4.41",
    "datasets":     "datasets>=2.19",
    "accelerate":   "accelerate>=0.30",
    "sacrebleu":    "sacrebleu",
    "sentencepiece":"sentencepiece",
    "mlflow":       "mlflow",
    "evaluate":     "evaluate",
}
missing = [spec for mod, spec in REQUIREMENTS.items()
           if importlib.util.find_spec(mod) is None]
if missing:
    print("Installing:", ", ".join(missing))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
    print("Done. If Colab shows a 'Restart session' prompt, restart and re-run this cell.")
else:
    print("All dependencies already present.")

All dependencies already present.


In [2]:
import os, time, json, random, math, glob
from pathlib import Path
import numpy as np
import pandas as pd

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch:", torch.__version__, "| device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU detected. Training will be slow — use DRY_RUN=True or a Colab GPU runtime "
          "(Runtime -> Change runtime type -> GPU).")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

torch: 2.13.0+cu130 | device: cuda
GPU: NVIDIA A100-SXM4-40GB


## 1 · Configuration

In [3]:
# ---- Paths ------------------------------------------------------------------
TRANSFORMED_CSV = Path("psa_main_transformed.csv")  # <-- output of the cleaning/EDA notebook
PROC_DIR   = Path("data_processed")          # per-direction split files are BUILT here
MODEL_DIR  = Path("models_week3");  MODEL_DIR.mkdir(exist_ok=True)
MLRUNS_DIR = Path("mlruns");        MLRUNS_DIR.mkdir(exist_ok=True)
RESULTS_DIR= Path("results_week3"); RESULTS_DIR.mkdir(exist_ok=True)

# ---- What to translate ------------------------------------------------------
# NLLB-200 language codes; Ekegusii is NOT in NLLB (hence None).
NLLB_CODE = {"English": "eng_Latn", "Kiswahili": "swh_Latn", "Ekegusii": None}

# Wide-table columns and the translation pairs to build from them.
LANG_COLS = ["English", "Kiswahili", "Ekegusii"]
PAIR_DIRECTIONS = [("English", "Kiswahili"),
                   ("English", "Ekegusii"),
                   ("Kiswahili", "Ekegusii")]
BIDIRECTIONAL = True     # also build the reverse of each pair (standard MT augmentation)
SPLIT_RATIOS  = (0.8, 0.1, 0.1)   # train / dev / test (grouped by PSA_ID)

# Direction slugs ("<Src>_to_<Tgt>") the models will train on (built in section 1b).
NLLB_DIRECTIONS = ["English_to_Kiswahili", "Kiswahili_to_English"]          # NLLB-supported
# LEAN: train only the two "into target" directions the demo uses. Add the others back
# (the 6-direction list is in the comment below) once you confirm a full run finishes.
MT5_DIRECTIONS  = ["English_to_Kiswahili", "English_to_Ekegusii"]
# Full set: ["English_to_Kiswahili","Kiswahili_to_English","English_to_Ekegusii",
#            "Ekegusii_to_English","Kiswahili_to_Ekegusii","Ekegusii_to_Kiswahili"]

# ---- Models -----------------------------------------------------------------
NLLB_NAME = "facebook/nllb-200-distilled-600M"
MT5_NAME  = "google/mt5-small"

# ---- Training hyperparameters (documented for the report) -------------------
DRY_RUN       = False    # full training. Set True only for a ~1-min plumbing smoke-test.
MAX_LEN       = 128
NUM_BEAMS     = 1        # greedy = far faster eval; bump to 4 just before the demo cell for nicer output
NO_REPEAT_NGRAM = 3      # block repeated 3-grams during generation
EVAL_SUBSET   = 400      # cap dev-set size used for per-epoch eval (speed); test eval stays full
FEWSHOT_N     = 4000     # cap train examples/direction (enough for a PoC; raise later if time allows)
FREEZE_ENCODER= False    # OFF by default: you have plenty of data, freezing hurts. Kept as an ablation arm (8a).

MT5_CFG  = dict(lr=1e-3, epochs=3, batch=16, optim="adafactor")  # mT5: Adafactor + higher LR
NLLB_CFG = dict(lr=2e-5, epochs=3, batch=8, optim="adamw_torch") # NLLB fine-tunes at a low LR (batch 8 = OOM-safe)

# ---- Experiment tracking ----------------------------------------------------
USE_WANDB = False        # set True and `wandb login` on Colab to also log to W&B
REPORT_TO = ["mlflow"] + (["wandb"] if USE_WANDB else [])

# ---- Scope switches (keep these OFF for a first, guaranteed-to-finish run) ----
DO_NLLB_FINETUNE = False   # NLLB zero-shot is already the strongest En<->Sw baseline; skip fine-tuning it
RUN_ABLATION     = False   # skip the extra freeze-vs-unfrozen run for now (turn on later for the report)

# ---- Persist outputs to Google Drive so a disconnect never loses trained models ----
USE_DRIVE = True
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        BASE = Path("/content/drive/MyDrive/psa_week3")
        MODEL_DIR   = BASE / "models_week3"
        RESULTS_DIR = BASE / "results_week3"
        PROC_DIR    = BASE / "data_processed"
        for d in (BASE, MODEL_DIR, RESULTS_DIR, PROC_DIR):
            d.mkdir(parents=True, exist_ok=True)
        print("Persisting models/results to:", BASE)
    except Exception as e:
        print("Drive not mounted (not on Colab?) — using local folders. Reason:", e)

if DRY_RUN:
    FEWSHOT_N = 40
    MT5_CFG["epochs"] = 1; NLLB_CFG["epochs"] = 1
    print(">>> DRY_RUN active: tiny data + 1 epoch (plumbing test only).")

Drive not mounted (not on Colab?) — using local folders. Reason: No module named 'google.colab'


## 1b · Build the training data from `psa_main_transformed.csv`

The cleaning/EDA notebook outputs one **wide** file (`PSA_ID, Domain, English, Kiswahili, Ekegusii, Data_Source`). This cell turns it into what the model sections need:

1. **Locate the file** — locally it sits next to the notebook; on Colab it prompts an upload.
2. **Leakage-safe splits** — train/dev/test assigned by **PSA_ID group** (stratified by Domain) so no announcement straddles splits.
3. **Melt into directional pairs** — English↔Kiswahili, English↔Ekegusii, Kiswahili↔Ekegusii (both directions), with NLLB language codes attached.
4. **Write** `data_processed/<Src>_to_<Tgt>.<split>.csv` (+ `psa_clean.csv`) — the exact format Section 2 onward loads. **No `data_processed.zip` needed anymore.**

In [4]:
import numpy as np

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
print("Environment:", "Google Colab" if IN_COLAB else "local Jupyter")

def _find_transformed():
    if TRANSFORMED_CSV.exists():
        return TRANSFORMED_CSV
    cands = (list(Path(".").glob("psa_main_transformed*.csv"))
             or list(Path(".").glob("*transformed*.csv")))
    return cands[0] if cands else None

# 1) Locate the wide cleaned CSV (upload on Colab if missing)
src_path = _find_transformed()
if src_path is None and IN_COLAB:
    from google.colab import files
    print("\nUpload psa_main_transformed.csv now:")
    files.upload()
    src_path = _find_transformed()
assert src_path is not None, (
    "psa_main_transformed.csv not found. Place it next to this notebook (or set TRANSFORMED_CSV), "
    "or upload it when prompted on Colab.")

wide = pd.read_csv(src_path)
wide.columns = [c.strip() for c in wide.columns]
for col in ["PSA_ID", "Domain"] + LANG_COLS:
    if col not in wide.columns:
        wide[col] = pd.NA
# blank strings -> missing, so coverage counts are honest
for c in LANG_COLS:
    wide[c] = wide[c].astype("string").str.strip().replace({"": pd.NA})
print(f"Loaded {len(wide):,} rows from {src_path.name}")
print("Coverage:", {c: int(wide[c].notna().sum()) for c in LANG_COLS})

# 2) Leakage-safe split by PSA_ID group, stratified by Domain
rng = np.random.default_rng(SEED)
tr_r, dv_r, _ = SPLIT_RATIOS
split_of_id = {}
for _dom, grp in wide.groupby("Domain"):
    ids = grp["PSA_ID"].dropna().astype(str).unique().tolist()
    rng.shuffle(ids)
    n = len(ids); n_tr = int(n * tr_r); n_dv = int(n * (tr_r + dv_r))
    for i, _id in enumerate(ids):
        split_of_id[_id] = "train" if i < n_tr else ("dev" if i < n_dv else "test")
wide["split"] = wide["PSA_ID"].astype(str).map(split_of_id).fillna("train")

# 3) Melt into directional pairs (both directions when BIDIRECTIONAL)
pair_rows = []
for _, r in wide.iterrows():
    for a, b in PAIR_DIRECTIONS:
        ta, tb = r[a], r[b]
        if isinstance(ta, str) and isinstance(tb, str) and ta and tb:
            dirs = [(a, ta, b, tb)]
            if BIDIRECTIONAL:
                dirs.append((b, tb, a, ta))
            for sl, st, tl, tt in dirs:
                pair_rows.append({
                    "src_text": st, "tgt_text": tt,
                    "src_code": NLLB_CODE.get(sl), "tgt_code": NLLB_CODE.get(tl),
                    "PSA_ID": r["PSA_ID"], "Domain": r["Domain"],
                    "pair": f"{sl}_to_{tl}", "split": r["split"],
                })
pairs = pd.DataFrame(pair_rows)
if len(pairs):
    pairs = pairs.drop_duplicates(subset=["pair", "src_text", "tgt_text"]).reset_index(drop=True)

# 4) Write per-direction split files (+ the wide clean table)
PROC_DIR.mkdir(exist_ok=True)
for f in PROC_DIR.glob("*_to_*.csv"):
    f.unlink()  # clear stale files from previous runs
for slug, g in pairs.groupby("pair"):
    for sp in ("train", "dev", "test"):
        gg = g[g["split"] == sp]
        if len(gg):
            gg[["src_text", "tgt_text", "src_code", "tgt_code", "PSA_ID", "Domain"]].to_csv(
                PROC_DIR / f"{slug}.{sp}.csv", index=False, encoding="utf-8-sig")
wide.to_csv(PROC_DIR / "psa_clean.csv", index=False, encoding="utf-8-sig")

made = sorted(p.name for p in PROC_DIR.glob("*_to_*.csv"))
print(f"\nWrote {len(made)} per-direction split files to {PROC_DIR}/  (+ psa_clean.csv)")
if len(pairs):
    print(pairs.groupby(["pair", "split"]).size().unstack(fill_value=0))
print("\nReady to train. (Ekegusii directions will be small — that is the low-resource case.)")


Environment: local Jupyter


Loaded 29,609 rows from psa_main_transformed.csv
Coverage: {'English': 29609, 'Kiswahili': 29609, 'Ekegusii': 29609}



Wrote 18 per-direction split files to data_processed/  (+ psa_clean.csv)
split                   dev  test  train
pair                                    
Ekegusii_to_English    2961  2963  23685
Ekegusii_to_Kiswahili  2961  2963  23685
English_to_Ekegusii    2961  2963  23685
English_to_Kiswahili   2960  2963  23677
Kiswahili_to_Ekegusii  2961  2963  23685
Kiswahili_to_English   2960  2963  23677

Ready to train. (Ekegusii directions will be small — that is the low-resource case.)


## 2 · Load the Week 2 data into Hugging Face datasets

Each Week 2 file (`<Src>_to_<Tgt>.<split>.csv`) already holds one direction with columns
`src_text, tgt_text, src_code, tgt_code, PSA_ID, Domain`. We wrap each direction in a
`DatasetDict` and, in the few-shot / low-resource setting, cap the training size to `FEWSHOT_N`.

In [5]:
from datasets import Dataset, DatasetDict

def load_direction(slug, few_shot_n=None):
    dsd = {}
    for sp in ("train", "dev", "test"):
        p = PROC_DIR / f"{slug}.{sp}.csv"
        if p.exists():
            df = pd.read_csv(p).dropna(subset=["src_text", "tgt_text"])
            if sp == "train" and few_shot_n:
                df = df.sample(min(len(df), few_shot_n), random_state=SEED).reset_index(drop=True)
            dsd[sp] = Dataset.from_pandas(df, preserve_index=False)
    return DatasetDict(dsd)

# Sanity: how many pairs per direction / split?
rows = []
for slug in MT5_DIRECTIONS:
    d = load_direction(slug)
    rows.append({"direction": slug, **{sp: (len(d[sp]) if sp in d else 0)
                                        for sp in ("train", "dev", "test")}})
avail = pd.DataFrame(rows).set_index("direction")
print(avail)

                      train   dev  test
direction                              
English_to_Kiswahili  23677  2960  2963
English_to_Ekegusii   23685  2961  2963


## 3 · Evaluation metrics (BLEU + chrF++)

We report **BLEU** and **chrF++** via `sacrebleu`. chrF++ is character-n-gram based and is the more
reliable signal for morphologically rich Bantu languages (Kiswahili, Ekegusii), so we select the
best checkpoint on chrF. (COMET and full human evaluation come in Week 4.)

In [6]:
import sacrebleu

def corpus_scores(preds, refs):
    preds = [p.strip() for p in preds]
    refs  = [r.strip() for r in refs]
    bleu = sacrebleu.corpus_bleu(preds, [refs]).score
    chrf = sacrebleu.corpus_chrf(preds, [refs], word_order=2).score   # word_order=2 => chrF++
    return {"bleu": round(bleu, 2), "chrf": round(chrf, 2)}

# Quick self-test of the metric on a trivial example
_demo = corpus_scores(["habari ya asubuhi"], ["habari ya asubuhi"])
print("metric self-test (perfect match):", _demo)

metric self-test (perfect match): {'bleu': 0.0, 'chrf': 100.0}


## 4 · Experiment tracking (MLflow)

In [7]:
import os
os.environ["MLFLOW_ENABLE_ASYNC_LOGGING"] = "false"  # avoid async_logging_queue errors on Colab
import mlflow
# Recent MLflow deprecated the plain file store; sqlite works everywhere (incl. Colab).
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("psa-mt-week3")
print("MLflow logging to sqlite:///mlflow.db")
# View later with:  !mlflow ui --backend-store-uri sqlite:///mlflow.db

2026/08/04 18:47:21 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/08/04 18:47:21 INFO mlflow.store.db.utils: Updating database tables


2026/08/04 18:47:23 INFO mlflow.tracking.fluent: Experiment with name 'psa-mt-week3' does not exist. Creating a new experiment.


MLflow logging to sqlite:///mlflow.db


## 5 · Shared model utilities

One place for the pieces both models share: encoder freezing (the low-resource trick),
a batched generation helper for zero-shot evaluation, and a `HF Trainer` factory.

In [8]:
from transformers import (AutoTokenizer, AutoModelForSeq2SeqLM,
                          Seq2SeqTrainer, Seq2SeqTrainingArguments,
                          DataCollatorForSeq2Seq)

def freeze_encoder(model):
    # Low-resource technique: freeze encoder weights so few examples can't overfit them.
    n = 0
    for p in model.get_encoder().parameters():
        p.requires_grad = False; n += p.numel()
    print(f"  froze encoder ({n/1e6:.1f}M params)")

def count_trainable(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

import re as _re
_SENTINEL = _re.compile(r"<extra_id_\d+>")
def _clean(s):
    # mT5 sometimes emits <extra_id_N> sentinel tokens; strip them from any output.
    return _SENTINEL.sub("", s).strip()

@torch.no_grad()
def generate_translations(model, tokenizer, texts, model_type,
                          src_lang, tgt_lang, batch_size=16, max_len=MAX_LEN):
    # Batched inference used for zero-shot eval and the demo. Returns list[str].
    model.eval()
    out = []
    for i in range(0, len(texts), batch_size):
        batch = list(texts[i:i + batch_size])
        if model_type == "nllb":
            tokenizer.src_lang = NLLB_CODE[src_lang]
            enc = tokenizer(batch, return_tensors="pt", padding=True,
                            truncation=True, max_length=max_len).to(model.device)
            forced = tokenizer.convert_tokens_to_ids(NLLB_CODE[tgt_lang])
            gen = model.generate(**enc, forced_bos_token_id=forced, max_length=max_len,
                                 num_beams=NUM_BEAMS, no_repeat_ngram_size=NO_REPEAT_NGRAM)
        else:  # mt5
            prompt = [f"translate {src_lang} to {tgt_lang}: {t}" for t in batch]
            enc = tokenizer(prompt, return_tensors="pt", padding=True,
                            truncation=True, max_length=max_len).to(model.device)
            gen = model.generate(**enc, max_length=max_len,
                                 num_beams=NUM_BEAMS, no_repeat_ngram_size=NO_REPEAT_NGRAM)
        out.extend(tokenizer.batch_decode(gen, skip_special_tokens=True))
    return [_clean(o) for o in out]

def make_compute_metrics(tokenizer):
    def compute_metrics(eval_preds):
        preds, labels = eval_preds
        if isinstance(preds, tuple): preds = preds[0]
        preds  = np.where(preds  != -100, preds,  tokenizer.pad_token_id)
        labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
        dpred = [_clean(x) for x in tokenizer.batch_decode(preds,  skip_special_tokens=True)]
        dref  = [_clean(x) for x in tokenizer.batch_decode(labels, skip_special_tokens=True)]
        return corpus_scores(dpred, dref)
    return compute_metrics

import inspect
def build_seq2seq_trainer(model, args, train_ds, eval_ds, collator, compute_metrics, tokenizer):
    # Recent transformers renamed Trainer's `tokenizer=` arg to `processing_class=`.
    kw = dict(model=model, args=args, train_dataset=train_ds, eval_dataset=eval_ds,
              data_collator=collator, compute_metrics=compute_metrics)
    params = inspect.signature(Seq2SeqTrainer.__init__).parameters
    kw["processing_class" if "processing_class" in params else "tokenizer"] = tokenizer
    return Seq2SeqTrainer(**kw)

def trainer_tokenizer(tr):
    # Fetch the tokenizer back off a Trainer across versions.
    return getattr(tr, "tokenizer", None) or getattr(tr, "processing_class", None)

## 6 · Model A — NLLB-200-distilled (English ↔ Kiswahili)

Two arms, which together form the **zero-shot vs few-shot** ablation:

1. **Zero-shot baseline** — the off-the-shelf model, no training. This is our reference point.
2. **Few-shot fine-tune** — same model, fine-tuned on the (capped) PSA training set, optionally with
   a frozen encoder.

In [9]:
RESULTS = []   # collected across all experiments -> ablation table in section 8

def eval_nllb_zero_shot(direction):
    src, tgt = direction.split("_to_")
    dsd = load_direction(direction)
    if "test" not in dsd or len(dsd["test"]) == 0:
        print("  no test data for", direction); return
    tok = AutoTokenizer.from_pretrained(NLLB_NAME)
    model = AutoModelForSeq2SeqLM.from_pretrained(NLLB_NAME).to(DEVICE)
    preds = generate_translations(model, tok, dsd["test"]["src_text"], "nllb", src, tgt)
    sc = corpus_scores(preds, dsd["test"]["tgt_text"])
    RESULTS.append({"model": "NLLB-distilled", "direction": direction,
                    "setting": "zero-shot", **sc, "n_test": len(dsd["test"])})
    with mlflow.start_run(run_name=f"nllb-zeroshot-{direction}"):
        mlflow.log_params({"model": NLLB_NAME, "setting": "zero-shot", "direction": direction})
        mlflow.log_metrics(sc)
    print(f"  NLLB zero-shot {direction}: {sc}")
    del model; torch.cuda.empty_cache() if DEVICE == "cuda" else None

for d in NLLB_DIRECTIONS:
    eval_nllb_zero_shot(d)

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 4.85MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.3MB            

...

model.safetensors: downloading bytes:           |  0.00B            

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

  NLLB zero-shot English_to_Kiswahili: {'bleu': 66.84, 'chrf': 80.25}


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

  NLLB zero-shot Kiswahili_to_English: {'bleu': 49.28, 'chrf': 69.92}


In [10]:
def preprocess_nllb(tokenizer, src, tgt):
    tokenizer.src_lang = NLLB_CODE[src]
    tokenizer.tgt_lang = NLLB_CODE[tgt]
    def fn(batch):
        enc = tokenizer(batch["src_text"], text_target=batch["tgt_text"],
                        max_length=MAX_LEN, truncation=True)
        return enc
    return fn

def finetune_nllb(direction, cfg=NLLB_CFG, freeze=FREEZE_ENCODER):
    src, tgt = direction.split("_to_")
    dsd = load_direction(direction, few_shot_n=FEWSHOT_N)
    if "train" not in dsd or len(dsd["train"]) == 0:
        print("  no train data for", direction); return None
    tok = AutoTokenizer.from_pretrained(NLLB_NAME)
    model = AutoModelForSeq2SeqLM.from_pretrained(NLLB_NAME).to(DEVICE)
    if freeze: freeze_encoder(model)

    enc = dsd.map(preprocess_nllb(tok, src, tgt), batched=True,
                  remove_columns=dsd["train"].column_names)
    out_dir = MODEL_DIR / f"nllb_{direction}"
    args = Seq2SeqTrainingArguments(
        output_dir=str(out_dir), learning_rate=cfg["lr"],
        per_device_train_batch_size=cfg["batch"], per_device_eval_batch_size=cfg["batch"],
        num_train_epochs=cfg["epochs"], weight_decay=0.0, optim=cfg["optim"],
        predict_with_generate=True, generation_max_length=MAX_LEN,
        generation_num_beams=NUM_BEAMS,
        fp16=(DEVICE == "cuda"), eval_strategy="epoch", save_strategy="epoch",
        save_total_limit=1, load_best_model_at_end=True, metric_for_best_model="chrf",
        greater_is_better=True, logging_steps=25, report_to=REPORT_TO,
        run_name=f"nllb-fewshot-{direction}",
    )   # older transformers: rename eval_strategy -> evaluation_strategy
    _dev = enc.get("dev", enc["train"])
    if EVAL_SUBSET and len(_dev) > EVAL_SUBSET:
        _dev = _dev.shuffle(seed=SEED).select(range(EVAL_SUBSET))
    trainer = build_seq2seq_trainer(
        model=model, args=args, train_ds=enc["train"],
        eval_ds=_dev,
        collator=DataCollatorForSeq2Seq(tok, model=model),
        compute_metrics=make_compute_metrics(tok), tokenizer=tok,
    )
    t0 = time.time(); trainer.train(); mins = (time.time() - t0) / 60

    test = enc.get("test")
    sc = trainer.evaluate(test) if test is not None else {}
    sc = {"bleu": round(sc.get("eval_bleu", float("nan")), 2),
          "chrf": round(sc.get("eval_chrf", float("nan")), 2)}
    RESULTS.append({"model": "NLLB-distilled", "direction": direction,
                    "setting": f"few-shot{'+freeze' if freeze else ''}",
                    **sc, "n_test": len(dsd.get("test", []))})
    trainer.save_model(str(out_dir)); tok.save_pretrained(str(out_dir))
    print(f"  NLLB few-shot {direction}: {sc} | {mins:.1f} min | "
          f"trainable={count_trainable(model)/1e6:.1f}M")
    return trainer

if DO_NLLB_FINETUNE:
    for d in NLLB_DIRECTIONS:
        finetune_nllb(d)
else:
    print('Skipping NLLB fine-tuning (DO_NLLB_FINETUNE=False); zero-shot baseline above is kept.')

Skipping NLLB fine-tuning (DO_NLLB_FINETUNE=False); zero-shot baseline above is kept.


## 7 · Model B — mT5-small (all directions, incl. low-resource Ekegusii)

mT5 is pretrained on span-corruption, not translation, so it is used **fine-tuned** (a zero-shot
mT5 arm is not meaningful — a useful contrast to note in the report). Because it is text-to-text,
it handles **Ekegusii** with no predefined language code — this is what makes the low-resource
directions possible.

In [11]:
def preprocess_mt5(tokenizer, src, tgt):
    def fn(batch):
        prompt = [f"translate {src} to {tgt}: {t}" for t in batch["src_text"]]
        enc = tokenizer(prompt, max_length=MAX_LEN, truncation=True)
        lab = tokenizer(text_target=batch["tgt_text"], max_length=MAX_LEN, truncation=True)
        enc["labels"] = lab["input_ids"]
        return enc
    return fn

def finetune_mt5(direction, cfg=MT5_CFG, freeze=FREEZE_ENCODER):
    src, tgt = direction.split("_to_")
    dsd = load_direction(direction, few_shot_n=FEWSHOT_N)
    if "train" not in dsd or len(dsd["train"]) == 0:
        print("  no train data for", direction); return None
    tok = AutoTokenizer.from_pretrained(MT5_NAME)
    model = AutoModelForSeq2SeqLM.from_pretrained(MT5_NAME).to(DEVICE)
    if freeze: freeze_encoder(model)

    enc = dsd.map(preprocess_mt5(tok, src, tgt), batched=True,
                  remove_columns=dsd["train"].column_names)
    out_dir = MODEL_DIR / f"mt5_{direction}"
    args = Seq2SeqTrainingArguments(
        output_dir=str(out_dir), learning_rate=cfg["lr"],
        per_device_train_batch_size=cfg["batch"], per_device_eval_batch_size=cfg["batch"],
        num_train_epochs=cfg["epochs"], weight_decay=0.0, optim=cfg["optim"],
        predict_with_generate=True, generation_max_length=MAX_LEN,
        generation_num_beams=NUM_BEAMS,
        fp16=False,  # mT5 is numerically unstable in fp16; keep fp32 (or use bf16 on Ampere+)
        eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
        load_best_model_at_end=True, metric_for_best_model="chrf", greater_is_better=True,
        logging_steps=25, report_to=REPORT_TO, run_name=f"mt5-fewshot-{direction}",
    )
    _dev = enc.get("dev", enc["train"])
    if EVAL_SUBSET and len(_dev) > EVAL_SUBSET:
        _dev = _dev.shuffle(seed=SEED).select(range(EVAL_SUBSET))
    trainer = build_seq2seq_trainer(
        model=model, args=args, train_ds=enc["train"],
        eval_ds=_dev,
        collator=DataCollatorForSeq2Seq(tok, model=model),
        compute_metrics=make_compute_metrics(tok), tokenizer=tok,
    )
    t0 = time.time(); trainer.train(); mins = (time.time() - t0) / 60
    test = enc.get("test")
    sc = trainer.evaluate(test) if test is not None else {}
    sc = {"bleu": round(sc.get("eval_bleu", float("nan")), 2),
          "chrf": round(sc.get("eval_chrf", float("nan")), 2)}
    RESULTS.append({"model": "mT5-small", "direction": direction,
                    "setting": f"few-shot{'+freeze' if freeze else ''}",
                    **sc, "n_test": len(dsd.get("test", []))})
    trainer.save_model(str(out_dir)); tok.save_pretrained(str(out_dir))
    print(f"  mT5 {direction}: {sc} | {mins:.1f} min")
    return trainer

mt5_trainers = {d: finetune_mt5(d) for d in MT5_DIRECTIONS}

config.json:   0%|          | 0.00/553 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.20GB            

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


Training Loss,Validation Loss,Epoch,Bleu,Chrf
2.919951,2.734953,3,3.680000,21.430000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  mT5 English_to_Ekegusii: {'bleu': 3.68, 'chrf': 21.43} | 6.0 min


## 8 · Ablation studies & initial performance summary

Three comparisons the rubric asks for:

- **Zero-shot vs few-shot** — already collected for NLLB (English↔Kiswahili).
- **Freeze vs full fine-tune** — re-run one direction with the encoder unfrozen and compare.
- **Per-domain** — break the best model's test score down by PSA domain (domain adaptation view).

In [12]:
# 8a. Freeze vs full fine-tune ablation.
# Main runs are unfrozen (FREEZE_ENCODER=False), so here we add the FROZEN arm to contrast.
ABLATION_DIR = "English_to_Kiswahili"
if RUN_ABLATION and not DRY_RUN:
    finetune_mt5(ABLATION_DIR, freeze=True)   # adds a "few-shot+freeze" row for contrast

# 8b. Assemble the results table
res = pd.DataFrame(RESULTS)
if len(res):
    res = res[["model", "direction", "setting", "bleu", "chrf", "n_test"]]
    res = res.sort_values(["direction", "model", "setting"]).reset_index(drop=True)
display(res)
res.to_csv(RESULTS_DIR / "performance_summary.csv", index=False)

,model,direction,setting,bleu,chrf,n_test
0,mT5-small,English_to_Ekegusii,few-shot,3.68,21.43,2963
1,NLLB-distilled,English_to_Kiswahili,zero-shot,66.84,80.25,2963
2,mT5-small,English_to_Kiswahili,few-shot,69.04,79.17,2963
3,NLLB-distilled,Kiswahili_to_English,zero-shot,49.28,69.92,2963


In [13]:
# 8c. Per-domain breakdown for the best mT5 direction we trained
def per_domain_eval(direction):
    tr = mt5_trainers.get(direction)
    if tr is None: return None
    dsd = load_direction(direction)
    if "test" not in dsd: return None
    test = dsd["test"].to_pandas()
    src, tgt = direction.split("_to_")
    preds = generate_translations(tr.model, trainer_tokenizer(tr), test["src_text"].tolist(),
                                  "mt5", src, tgt)
    test["pred"] = preds
    out = []
    for dom, g in test.groupby("Domain"):
        sc = corpus_scores(g["pred"].tolist(), g["tgt_text"].tolist())
        out.append({"direction": direction, "domain": dom, **sc, "n": len(g)})
    return pd.DataFrame(out)

pd_res = per_domain_eval("English_to_Kiswahili")
if pd_res is not None:
    display(pd_res)
    pd_res.to_csv(RESULTS_DIR / "per_domain_english_kiswahili.csv", index=False)

,direction,domain,bleu,chrf,n
0,English_to_Kiswahili,Agriculture,70.81,80.90,1411
1,English_to_Kiswahili,Education,70.06,80.36,416
2,English_to_Kiswahili,Governance,70.45,80.93,345
3,English_to_Kiswahili,Health,64.28,75.70,396
4,English_to_Kiswahili,Security & Safety,61.12,74.05,395


## 9 · Save logs & summary

In [14]:
summary = {
    "device": DEVICE, "dry_run": DRY_RUN, "few_shot_n": FEWSHOT_N,
    "freeze_encoder": FREEZE_ENCODER, "max_len": MAX_LEN,
    "mt5_cfg": MT5_CFG, "nllb_cfg": NLLB_CFG,
    "results": RESULTS,
}
with open(RESULTS_DIR / "week3_summary.json", "w") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print("Saved:")
for p in sorted(RESULTS_DIR.glob("*")): print("  -", p)
print("Checkpoints under:", MODEL_DIR.resolve())
print("MLflow runs      : sqlite:///mlflow.db  (mlflow ui --backend-store-uri sqlite:///mlflow.db)")

Saved:
  - results_week3/per_domain_english_kiswahili.csv
  - results_week3/performance_summary.csv
  - results_week3/week3_summary.json
Checkpoints under: /root/work/models_week3
MLflow runs      : sqlite:///mlflow.db  (mlflow ui --backend-store-uri sqlite:///mlflow.db)


## 10 · Inference demo (Success criterion: a working translation demo)

A single `translate()` entry point that loads a saved checkpoint and translates a PSA. The same
logic is written out to **`translate_psa.py`** so it doubles as the Week 3 command-line deliverable.

In [15]:
from functools import lru_cache

@lru_cache(maxsize=4)
def _load(model_path, model_type):
    tok = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_path).to(DEVICE)
    return tok, model

def translate(text, src_lang, tgt_lang, model_path, model_type="mt5"):
    tok, model = _load(model_path, model_type)
    return generate_translations(model, tok, [text], model_type, src_lang, tgt_lang)[0]

# Example (after training; picks the mT5 English->Kiswahili checkpoint if present)
_ckpt = MODEL_DIR / "mt5_English_to_Kiswahili"
if _ckpt.exists():
    demo = "Ministry of Health urges residents to complete their vaccination before Friday."
    print("EN :", demo)
    print("SW :", translate(demo, "English", "Kiswahili", str(_ckpt), "mt5"))
else:
    print("Train a model first (run sections 6–7), then re-run this cell.")

EN : Ministry of Health urges residents to complete their vaccination before Friday.


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


SW : Wizara ya Afya inawasihi wakazi wa Kaunti ya Mimea kukamilisha maombi ya uzazi kabla ya tarehe ya mwisho.


In [16]:
%%writefile translate_psa.py
# Week 3 CLI deliverable: python translate_psa.py --text "..." --src English --tgt Kiswahili --model models_week3/mt5_English_to_Kiswahili
import argparse, torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

NLLB_CODE = {"English": "eng_Latn", "Kiswahili": "swh_Latn", "Ekegusii": None}
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def translate(text, src, tgt, model_path, model_type):
    tok = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_path).to(DEVICE)
    if model_type == "nllb":
        tok.src_lang = NLLB_CODE[src]
        enc = tok(text, return_tensors="pt", truncation=True, max_length=128).to(DEVICE)
        gen = model.generate(**enc, forced_bos_token_id=tok.convert_tokens_to_ids(NLLB_CODE[tgt]),
                             max_length=128)
    else:
        enc = tok(f"translate {src} to {tgt}: {text}", return_tensors="pt",
                  truncation=True, max_length=128).to(DEVICE)
        gen = model.generate(**enc, max_length=128)
    return tok.batch_decode(gen, skip_special_tokens=True)[0].strip()

if __name__ == "__main__":
    ap = argparse.ArgumentParser(description="Translate a PSA between English/Kiswahili/Ekegusii.")
    ap.add_argument("--text", required=True)
    ap.add_argument("--src", required=True)
    ap.add_argument("--tgt", required=True)
    ap.add_argument("--model", required=True, help="path to a saved checkpoint")
    ap.add_argument("--model_type", default="mt5", choices=["mt5", "nllb"])
    a = ap.parse_args()
    print(translate(a.text, a.src, a.tgt, a.model, a.model_type))

Writing translate_psa.py


## 11 · Summary & hand-off to Week 4 (Evaluation & Deployment)

**Delivered this week**

- Two fine-tuned model families (NLLB-distilled for English↔Kiswahili, mT5-small for all
  directions including Ekegusii), tracked in MLflow with saved checkpoints under `models_week3/`.
- Ablations: zero-shot vs few-shot, freeze vs full fine-tune, and a per-domain breakdown.
- `results_week3/performance_summary.csv` (the initial performance summary) and a working
  `translate()` demo + `translate_psa.py` CLI.

**What the numbers will tell you**

- English↔Kiswahili should be strong (NLLB already knows both; fine-tuning adapts it to PSA style).
- Ekegusii will lag — expected, and exactly the low-resource gap the project studies. Note in the
  report whether encoder-freezing and the few-shot cap helped or hurt each direction.

**Week 4 next steps**

1. Add **COMET** (`unbabel-comet`) alongside BLEU/chrF, and run **human evaluation** (fluency /
   adequacy / cultural accuracy) on 100+ sentences using the Week 2 native-speaker subset.
2. **Error analysis** per domain and per direction; document limitations.
3. Wrap the best checkpoints in a **Streamlit/Gradio** app (input PSA → choose target language →
   translation + confidence + feedback form).
4. Finalise the GitHub repo: code, dataset link, notebooks, README, CC-BY license.

**Troubleshooting (Colab)**

- OOM → lower `batch`, keep `MAX_LEN=128`, or use `google/mt5-small` only.
- mT5 loss stuck at 0 / NaN → keep `fp16=False` (already set); mT5 is unstable in fp16.
- Slow → confirm GPU runtime; start with `DRY_RUN=True` to verify the pipeline end-to-end.

## 12 · Predict on sample PSAs

Runs the trained models on a handful of example PSAs and shows English → **Kiswahili** and English → **Ekegusii** side by side. By default it uses the strongest model per language: **NLLB** for Kiswahili (its zero-shot was the best English↔Kiswahili result) and **mT5** for Ekegusii (NLLB can't do Ekegusii). Edit `SAMPLE_PSAS` to try your own, or point a target at a different checkpoint. Saved to `results_week3/sample_psa_predictions.csv`.

> Run this after training (sections 6–7). Decoding uses beam search and strips mT5 sentinel tokens.

In [17]:
# Example PSAs to translate (edit freely)
SAMPLE_PSAS = [
    "Ministry of Health urges residents to complete their vaccination before Friday.",
    "IEBC reminds voters to verify their registration details via SMS before the deadline.",
    "Farmers are advised to collect subsidised fertiliser at the nearest depot this week.",
    "Schools reopen on Monday; parents should ensure all fees are cleared early.",
    "Avoid unnecessary travel to flood-affected areas until further notice.",
]

# Best model per target language. Swap Kiswahili to the mT5 checkpoint if you prefer:
#   "Kiswahili": (MODEL_DIR / "mt5_English_to_Kiswahili", "mt5")
TARGETS = {
    "Kiswahili": (NLLB_NAME, "nllb"),                              # NLLB: strongest for En<->Sw
    "Ekegusii":  (MODEL_DIR / "mt5_English_to_Ekegusii", "mt5"),   # mT5: only option for Ekegusii
}

def _usable(path):
    p = str(path)
    return Path(p).exists() or ("/" in p and not p.startswith(("models_week3", "data_processed")))

rows = []
for text in SAMPLE_PSAS:
    row = {"English": text}
    for tgt, (path, mtype) in TARGETS.items():
        if _usable(path):
            try:
                row[tgt] = translate(text, "English", tgt, str(path), mtype)
            except Exception as e:
                row[tgt] = f"(error: {e})"
        else:
            row[tgt] = f"(no {tgt} checkpoint — run training first)"
    rows.append(row)

preds = pd.DataFrame(rows)[["English", "Kiswahili", "Ekegusii"]]
RESULTS_DIR.mkdir(exist_ok=True)
preds.to_csv(RESULTS_DIR / "sample_psa_predictions.csv", index=False, encoding="utf-8-sig")

for _, r in preds.iterrows():
    print("=" * 72)
    print("EN :", r["English"])
    print("SW :", r["Kiswahili"])
    print("EK :", r["Ekegusii"])

preds


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


EN : Ministry of Health urges residents to complete their vaccination before Friday.
SW : Wizara ya Afya inawaomba wakazi kukamilisha chanjo yao kabla ya Ijumaa.
EK : Ekeombe gia Oboremi na Oborema nigo korigiriria abanto barengete abanto gochia gochia ase ekaunti ya ekaonti ya ebirengererio bia abana babo ase omotienyi 5.
EN : IEBC reminds voters to verify their registration details via SMS before the deadline.
SW : IEBC inawakumbusha wapiga kura kuthibitisha maelezo yao ya usajili kupitia SMS kabla ya tarehe ya mwisho.
EK : IEBC egokora abamenyi ba aaria ekaunti ya ekaonti ya ekeyia ya ao aao goetera chinchera chikura.
EN : Farmers are advised to collect subsidised fertiliser at the nearest depot this week.
SW : Wakulima wanashauriwa kukusanya mbolea za ziada kwenye ghala la karibu zaidi wiki hii.
EK : Ekeombe keria gekorigereria abaremi baborigwe aaria ekaonti ya ekaunti ya ebirio ao ekeyia amoyo a ekao ekaonsi ekaomwaka ekaone ekaenerete ekaonchoka ekaao ase omotienyi 6.
EN : Schoo

,English,Kiswahili,Ekegusii
0,Ministry of Health urges residents to complete...,Wizara ya Afya inawaomba wakazi kukamilisha ch...,Ekeombe gia Oboremi na Oborema nigo korigiriri...
1,IEBC reminds voters to verify their registrati...,IEBC inawakumbusha wapiga kura kuthibitisha ma...,IEBC egokora abamenyi ba aaria ekaunti ya ekao...
2,Farmers are advised to collect subsidised fert...,Wakulima wanashauriwa kukusanya mbolea za ziad...,Ekeombe keria gekorigereria abaremi baborigwe ...
3,Schools reopen on Monday; parents should ensur...,Shule hufungua tena Jumatatu; wazazi wanapaswa...,Ekeombe keria gekorigereria abana bare gochia ...
4,Avoid unnecessary travel to flood-affected are...,Epuka kusafiri bila sababu kwenda maeneo yaliy...,Ekere okoigora okogesa amache okobwaterana oko...
